# Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 120

# Prepare Jets and B-Quarks

## Load Data

In [31]:
JETS_PATH = 'ak4_jets.csv'
QUARKS_PATH = 'b_quarks.csv'
THRESHOLD = 30  # percent
PT_MIN = 15     # GeV
ETA_MAX = 2.5

jets = pd.read_csv(JETS_PATH)
jets = jets[jets['truth_flavor'] == 'b'].copy()                             # Keep only b jets for the analysis
jets = jets[(jets['pt'] > PT_MIN) & (jets['eta'].abs() < ETA_MAX)].copy()   # Apply kinematic cuts
jets.drop(columns=['hepmc_event', 'y', 'px', 'py', 'pz', 'E', 'truth_flavor', 'btag_loose', 'btag_medium', 'btag_tight'], inplace=True)

quarks = pd.read_csv(QUARKS_PATH)
quarks.drop(columns=['hepmc_event', 'y', 'px', 'py', 'pz', 'E',], inplace=True)

jets.reset_index(drop=True, inplace=True)
quarks.reset_index(drop=True, inplace=True)

## Jets Higgs Fraction

In [32]:
# Define Higgs fraction as a percent of jet constituents that come from H1, H2, or H3
jets['f_H'] = jets[['H1_percentage', 'H2_percentage', 'H3_percentage']].sum(axis=1)

# Prune jets that have less Higgs origin than threshold
jets = jets[jets['f_H'] >= THRESHOLD]
jets.reset_index(drop=True, inplace=True)

In [33]:
# Conditioned fractions h_i = H_i / f_H
jets["h1"] = jets["H1_percentage"] / jets["f_H"]
jets["h2"] = jets["H2_percentage"] / jets["f_H"]
jets["h3"] = jets["H3_percentage"] / jets["f_H"]

In [ ]:
# Define pureness as p = max(h1, h2, h3)
jets["pureness"] = jets[["h1", "h2", "h3"]].max(axis=1)

# Define mixing entropy
h_cols = ["h1", "h2", "h3"]
eps = 1e-12
jets["S_H"] = - (jets[h_cols] * np.log(jets[h_cols] + eps)).sum(axis=1)

,event,jet,pt,eta,phi,mass,num_constituents,H1_percentage,H2_percentage,H3_percentage,f_H,h1,h2,h3,pureness,S_H
0,0,1,373.374573,-1.244345,-3.137681,35.930534,35,2.857143,91.428571,0.000000,94.285714,0.030303,0.969697,0.000000,0.969697,1.357940e-01
1,0,3,104.091063,-0.900220,0.469602,10.316478,24,4.166667,0.000000,87.500000,91.666667,0.045455,0.000000,0.954545,0.954545,1.849074e-01
2,0,4,69.651185,-0.253178,-1.339856,8.128798,20,85.000000,0.000000,0.000000,85.000000,1.000000,0.000000,0.000000,1.000000,-1.000089e-12
3,0,5,42.220545,-0.524350,0.822800,6.104056,23,0.000000,0.000000,69.565217,69.565217,0.000000,0.000000,1.000000,1.000000,-1.000089e-12
4,0,6,25.622103,-1.399052,2.595420,5.509801,17,58.823529,5.882353,5.882353,70.588235,0.833333,0.083333,0.083333,0.833333,5.660857e-01
